In [2]:
import pickle
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    SimpleRNN,
    Dense,
    Input
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping

# Load Dataset

df = pd.read_csv("IMDB Dataset.csv")

print(df.head())
print(df.shape)

# Convert labels
df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

X = df["review"].astype(str).values
y = df["sentiment"].values

# Train Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Text Cleaning

def custom_standardization(text):

    text = tf.strings.lower(text)

    text = tf.strings.regex_replace(text, "<br />", " ")

    text = tf.strings.regex_replace(
        text,
        "[^a-zA-Z ]",
        ""
    )

    return text

# Text Vectorization

VOCAB_SIZE = 10000
MAX_LEN = 200

vectorizer = TextVectorization(
    standardize=custom_standardization,
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN
)

vectorizer.adapt(X_train)

# Save vocabulary
vocab = vectorizer.get_vocabulary()

with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

# Vectorize Dataset

X_train = vectorizer(tf.constant(X_train))
X_test = vectorizer(tf.constant(X_test))

# Build Model

model = Sequential([
    Input(shape=(MAX_LEN,)),

    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        mask_zero=True
    ),

    SimpleRNN(64),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0005
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# Training

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    callbacks=[early_stop]
)

# Evaluation

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"\nTest Accuracy: {accuracy:.4f}")

# Save Model

model.save("imdb_model.keras")

print("\nModel Saved Successfully!")
print("Vocabulary Saved Successfully!")

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,292,417 (4.93 MB)

 Trainable params: 1,292,417 (4.93 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 83s 76ms/step - accuracy: 0.6338 - loss: 0.6359 - val_accuracy: 0.6584 - val_loss: 0.6144
Epoch 2/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 71s 71ms/step - accuracy: 0.7370 - loss: 0.5334 - val_accuracy: 0.7561 - val_loss: 0.5249
Epoch 3/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 72s 60ms/step - accuracy: 0.8660 - loss: 0.3188 - val_accuracy: 0.7906 - val_loss: 0.5063
Epoch 4/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 79s 79ms/step - accuracy: 0.9629 - loss: 0.1131 - val_accuracy: 0.8031 - val_loss: 0.6106
Epoch 5/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 59s 59ms/step - accuracy: 0.9897 - loss: 0.0375 - val_accuracy: 0.7623 - val_loss: 0.8280
Epoch 6/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 76s 76ms/step - accuracy: 0.9946 - loss: 0.0198 - val_accuracy: 0.7343 - val_loss: 0.9951
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - accuracy: 0.7894 - loss: 0.4971

Test Accuracy: 0.7894

Model Saved Successfully!
Vocabulary Saved Successfully!
